# Parte 1: Limpieza, Exploración y Modificaciones (10 Ejercicios Básicos/Intermedios)
Para los ejercicios 1 y 2, debes asumir que estás trabajando sobre una tabla de prueba llamada *tb_reservas_prueba*.


Ojo: Para evitar poner a cada rato *cursodataacademy.hotel_db.tb_reservas*, ejecutamos primero estas dos instrucciones:

In [0]:
%sql
USE CATALOG cursodataacademy;
USE SCHEMA hotel_db;

Comprobamos

In [0]:
%sql
SELECT *
FROM tb_reservas
LIMIT 10;

## Ejercicio 1. CREATE / INSERT

In [0]:
%sql
-- Crear tabla de prueba vacía
CREATE OR REPLACE TABLE tb_reservas_prueba
USING DELTA
AS
SELECT *
FROM tb_reservas
WHERE 1 = 0;

In [0]:
%sql
-- Insertar reservas canceladas del City Hotel
INSERT INTO tb_reservas_prueba
SELECT *
FROM tb_reservas
WHERE hotel = 'City Hotel'
  AND is_canceled = 1;

In [0]:
%sql
-- Comprobamos la inserción
SELECT
    hotel,
    is_canceled,
    COUNT(*) AS total_reservas
FROM tb_reservas_prueba
GROUP BY hotel, is_canceled;

**Conclusión:** Se ha construido una tabla de prueba que contiene exclusivamente las reservas canceladas de City Hotel, permitiendo estudiar este segmento sin alterar los registros originales.

## Ejercicio 2. UPDATE

In [0]:
%sql
UPDATE tb_reservas_prueba
SET babies = 0;

In [0]:
%sql
-- Verificamos
SELECT
    COUNT(*) AS total_reservas,
    SUM(babies) AS total_bebes
FROM tb_reservas_prueba;

**Conclusión:** Se corrigen los registros afectados por el error de captura en la tabla de prueba, evitando que este problema distorsione el análisis de huéspedes con bebés.

## Ejercicio 3. CAST / CONCAT

In [0]:
%sql
SELECT
    arrival_date_day_of_month,
    arrival_date_month,
    arrival_date_year,

    CONCAT(
        LPAD(CAST(arrival_date_day_of_month AS STRING), 2, '0'),
        '-',
        arrival_date_month,
        '-',
        CAST(arrival_date_year AS STRING)
    ) AS arrival_date_string

FROM tb_reservas_prueba;

**Conclusión:** La transformación facilita la interpretación de las fechas de llegada, permitiendo presentar información temporal de manera uniforme y legible.

## Ejercicio 4. LIKE

In [0]:
%sql

SELECT *
FROM tb_reservas_prueba
WHERE distribution_channel LIKE 'T%';

**Conclusión:** El filtro permite identificar las reservas asociadas a canales de distribución que comienzan con T y delimitar segmentos para posteriores análisis comerciales.

## Ejercicio 5. COUNT / SUM / AVG

In [0]:
%sql
SELECT
    COUNT(*) AS total_reservas,
    SUM(total_of_special_requests) AS total_solicitudes_especiales,
    ROUND(AVG(adr), 2) AS precio_promedio_noche
FROM tb_reservas_prueba;

**Conclusión:** Estos indicadores permiten conocer el volumen de operaciones, la demanda de servicios adicionales y el precio promedio por noche, proporcionando una visión general del negocio hotelero.

## Ejercicio 6. DELETE

In [0]:
%sql
DELETE FROM tb_reservas_prueba
WHERE adr = 0.0;

In [0]:
%sql
-- Verificación
SELECT
    COUNT(*) AS reservas_restantes,
    COUNT_IF(adr = 0.0) AS reservas_precio_cero
FROM tb_reservas_prueba;

**Conclusión:** La eliminación de registros con tarifas iguales a cero permite depurar la muestra de prueba, excluyendo las reservas consideradas operaciones de prueba del sistema.

## Ejercicio 7. GROUP BY

In [0]:
%sql
SELECT
    distribution_channel,
    COUNT(*) AS total_reservas
FROM tb_reservas_prueba
GROUP BY distribution_channel
ORDER BY total_reservas DESC;

**Conclusión:** La distribución de reservas por canal permite identificar cuáles concentran la mayor actividad comercial y dónde conviene profundizar el análisis de cancelaciones.

## Ejercicio 8. CASE WHEN

In [0]:
%sql
SELECT
    hotel,
    adr,

    CASE
        WHEN adr < 50 THEN 'Económico'
        WHEN adr <= 150 THEN 'Estándar'
        ELSE 'Premium'
    END AS categoria_precio
FROM tb_reservas_prueba;

**Conclusión:** La segmentación por tarifas permite diferenciar las reservas según su valor económico y facilita posteriores análisis sobre el comportamiento de cancelación de cada categoría.

## Ejercicio 9. CASE WHEN

In [0]:
%sql

WITH reservas_fechas AS (
    SELECT
        lead_time,
        MAKE_DATE(
            CAST(arrival_date_year AS INT),
            CASE arrival_date_month
                WHEN 'January' THEN 1
                WHEN 'February' THEN 2
                WHEN 'March' THEN 3
                WHEN 'April' THEN 4
                WHEN 'May' THEN 5
                WHEN 'June' THEN 6
                WHEN 'July' THEN 7
                WHEN 'August' THEN 8
                WHEN 'September' THEN 9
                WHEN 'October' THEN 10
                WHEN 'November' THEN 11
                WHEN 'December' THEN 12
            END,
            CAST(arrival_date_day_of_month AS INT)
        ) AS arrival_date

    FROM tb_reservas_prueba
)

SELECT
    arrival_date,
    lead_time,
    DATE_SUB(
        arrival_date,
        CAST(lead_time AS INT)
    ) AS booking_date

FROM reservas_fechas;

**Conclusión:** El cálculo permite estimar cuándo se realizó cada reserva, facilitando el análisis de los patrones de anticipación de los clientes.

## Ejercicio 10. Funciones matemáticas CEIL

In [0]:
%sql
SELECT
    lead_time,
    CEIL(
        lead_time / 30.0
    ) AS meses_anticipacion_techo

FROM tb_reservas_prueba;

**Conclusión:** La conversión de días a meses aproximados facilita la segmentación de las reservas según su anticipación y permite identificar posibles patrones de comportamiento.